<a href="https://colab.research.google.com/github/rubusarbaro/supplychain-forecast-FIME/blob/main/Evaluaciones/PIA_HoltWinter_conExogenos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##################################################
##                                              ##
##             PRODUCTO INTEGRADOR              ##
##    Pronósticos en la Cadena de Suministro    ##
##                                              ##
##  Método: Holt-Winter                         ##
##                                              ##
##  Saúl Roberto Morales Velázquez              ##
##  1856691                                     ##
##                                              ##
##################################################

#**Librerías**

In [ ]:
### Librerías de Google para uso en Colab.
from google.colab import files, userdata  # Permite utilizar los datos de usuario.

### Librerías para trabajar con los datos
import numpy as np # Permite trabajar con opercaciones matemáticas avanzadas.
import pandas as pd # Permite trabajar con data frames.
from sklearn.metrics import mean_absolute_percentage_error as MAPE # Métricas de evaluación.

### Librerías para generar gráficas
import plotly.express as px # Librería que permite la creación de gráficas interactivas.
import plotly.graph_objects as go # Librería que permite la creación de gráficas interactivas.

### Librería del modelo a utilizar
from statsmodels.tsa.holtwinters import ExponentialSmoothing  # Librería de Holt-Winter, aunque el modelo a utilizar

### Otras librerías
import warnings
warnings.filterwarnings('ignore')

#**Funciones**

In [ ]:
# @title Gráfica del conjunto de datos real

def real_plt(df: object, time_column_name: str, value_column_name: str, title: str, xaxis_title="Tiempo", yaxis_title="Eje y") :
  """
  Grafica la serie de tiempo con los datos reales.

  Args:
      df (object): DataFrame que contiene los datos a graficar.
      time_column_name (str): Nombre de la columna que contiene las fechas.
      value_column_name (str): Nombre de la columna que contiene los valores.
      title (str): Título de la gráfica.

  Returns:
      Object: Gráfica de la serie de tiempo.
  """

  fig = px.line(df, x=time_column_name, y=value_column_name, title=title)

  fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
      buttons=list([
        dict(count=1, label="1m", step="month", stepmode="backward"),
        dict(count=6, label="6m", step="month", stepmode="backward"),
        dict(count=1, label="YTD", step="year", stepmode="todate"),
        dict(count=1, label="1y", step="year", stepmode="backward"),
        dict(step="all")
      ])
    )
  )

  fig.update_layout(
    title = title,
    xaxis_title = xaxis_title,
    yaxis_title = yaxis_title
  )

  return fig.show()

In [ ]:
# @title Gráfica comparativa del conjunto de datos real vs. el pronosticado

def real_vs_forecast_plt(real_df: object, real_time_column_name: str, real_value_column_name: str, forecast_df: object, title: str, forecast_value_column_name = 0, model_name_serie="Forecast", xaxis_title = "Tiempo", yaxis_title = "Eje y") :
  """
  Grafica la comparativa de las series de tiempo con los datos reales y pronosticados.

  Args:
      real_df (object): DataFrame que contiene los datos reales.
      real_time_column_name (str): Nombre de la columna que contiene las fechas de los datos reales.
      real_value_column_name (str): Nombre de la columna que contiene los valores reales.
      forecast_df (object): DataFrame que contiene los datos pronosticados.
      title (str): Título de la gráfica.
      forecast_time_column_name (str): Nombre de la columna que contiene las fechas de los datos pronosticados.
      forecast_value_column_name (str): Nombre de la columna que contiene los valores pronosticados.

  Returns:
      Object: Gráfica de la serie de tiempo.
  """

  fig = go.Figure()

  fig.add_trace(
      go.Scatter(
          x = forecast_df.index,
          y = forecast_df[forecast_value_column_name],
          name = model_name_serie,
          line = dict(color = "#8C000F", width = 3, dash = "dash")
      )
  )

  fig.add_trace(
      go.Scatter(
          x = real_df.index,
          y = real_df[real_value_column_name],
          name = "Real",
          line = dict(color = "olive", width = 3, dash = "dash")
      )
  )

  fig.update_layout(title = title, xaxis_title=xaxis_title, yaxis_title=yaxis_title)

  return fig.show()

#**Clases**

In [ ]:
# @title Banxico restAPI

class banxico() :
  """
  Clase que permite trabajar con los datos abiertos obtenidos mediante la API de Banxico.

  Args:
      token (str): Token de Banxico.
  """
  def __init__(self, token: str) :
    self._token = token  # Token de la API de Banxico. El uso del guión bajo permite establecer que es una variable interna, por lo que no puede ser llamada fuera de la clase.
    self.serie_ID = "The serie ID hasn't been defined yet."  # Última serie de datos utilizada en el modelo.
    self.url = "The serie ID hasn't been defined yet." # URL con la última serie de datos utilizada en el modelo.
    self.response = "The serie ID hasn't been defined yet."  # Respuesta HTTP de la última petición.
    self.HTTP_status_code = 0 # Código de estatus HTTP de la última petición.
    self.HTTP_status = "The serie ID hasn't been defined yet." # Estatus HTTP legible por humanos de la última petición.
    self.json_data = "The serie ID hasn't been defined yet."  # Datos en formato JSON utilizados para la construcción del conjunto de datos.
    self.title = "The serie ID hasn't been defined yet."  # Título del conjunto de datos, obtenido de la petición HTTP original.
    self.df = pd.DataFrame()  # Conjunto de datos.

  def get_df(self, serie_ID: str) :
    """
    Obtiene los datos de la serie de Banxico en formato de conjunto de datos (data frame).

    Args:
        serie_ID (str): ID de la serie de datos.

    Returns:
        Object: Conjunto de datos.
    """

    ### Librerías usadas en esta clase
    import json # Permite trabajar con datos en formato JSON.
    import requests # Permite realizar peticiones HTML.

    HTTP_codes = {
      "200" : "OK",
      "400" : "Bad Request",
      "401" : "Unauthorized",
      "403" : "Forbidden",
      "404" : "Not Found",
      "500" : "Internal Server Error",
    } # Diccionario de estatus HTTP.

    self.url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{serie_ID}/datos?token={self._token}" # Construye la URL del API para la serie de datos y la almacena en la variable del objeto.

    self.response = requests.get(self.url)  # Petición HTML.
    self.HTTP_status_code = self.response.status_code # Almacena el código de la petición HTML.
    self.HTTP_status = HTTP_codes[str(self.HTTP_status_code)] # Almacena el significado del código HTML.

    self.json_data = self.response.json() # Petición en formato JSON.
    self.title = self.json_data["bmx"]["series"][0]["titulo"] # Extrae el título de la serie de datos del JSON y la almacena en una variable del objeto.
    self.df = pd.DataFrame(self.json_data["bmx"]["series"][0]["datos"]) # Extrae los datos del JSON y los almacena en un conjunto de datos.
    self.df["fecha"] = pd.to_datetime(self.df["fecha"], format="%d/%m/%Y")  # Convierte las fechas en formato STR en formato datetime.

    if self.HTTP_status_code == 200 : # Revisa si la petición es correcta.
      return self.df  # Si es correcta, retorna el conjunto de datos.
    else :  # De lo contrario…
      print(f"Ha ocurrido un error: ({self.HTTP_status_code}) {self.HTTP_status}")  # Imprime el código de error
      return None # Y no retorna nada

#**Conjunto de datos**

In [ ]:
# @title Parámetros de obtención de datos

## Solo muestra Banxico, ya que no tengo ninguna otra fuente de datos.
Fuente = "Banxico" # @param ["Banxico"]

## Series de tiempo de Banxico que he precargado.
Datos = "Tipo de cambio MXN/USD" # @param ["Tipo de cambio MXN/USD","Monto de ventas CETES 28","Tasa de descuento CETES 28"]

series_dict = {
    "Tipo de cambio MXN/USD" : "SF43718",
    "Monto de ventas CETES 28" : "SF116765",
    "Tasa de descuento CETES 28" : "SF116766"
} # Diccionario con las series de tiempo y las descripciones mostradas anteriormente.

banxico_serieID = series_dict[Datos]  # Busca el id de la serie de tiempo en el diccionario.

data = banxico(userdata.get('Banxico_Token')) # Usa la clase banxico para interactuar con la API.
df = data.get_df(banxico_serieID) # Extrae la serie de tiempo elegida.
df.head() # Muestra el encabezado de la serie de tiempo.

,fecha,dato
0,1991-11-12,3.0735
1,1991-11-13,3.0712
2,1991-11-14,3.0718
3,1991-11-15,3.0684
4,1991-11-18,3.0673


In [ ]:
# @title Gráfica del conjunto de datos real
real_plt(df, "fecha", "dato", data.title, yaxis_title="Tipo de cambio")

In [ ]:
df4model = df.copy()  # Copia el conjunto de datos original en uno nuevo para alimentar el modelo.
df4model["dato"] = pd.to_numeric(df["dato"], errors="coerce") # Convierte los datos string de Banxico en datos numéricos int64.
df4model.set_index("fecha", inplace=True) # Convierte el índice INT en un índice datetime.

df4model.head() # Muestra las primeras filas del conjunto de datos.

,dato
fecha,
1991-11-12,3.0735
1991-11-13,3.0712
1991-11-14,3.0718
1991-11-15,3.0684
1991-11-18,3.0673


In [ ]:
fechas_completas = pd.date_range(start=df4model.index.min(), end=df4model.index.max(), freq='D')  # Rellena las fechas faltantes en el índice.
fechas_faltantes = fechas_completas.difference(df4model.index)  # Busca las fechas faltantes y las almacena en una lista.
print(f"Fechas faltantes: {fechas_faltantes}")  # Imprime la lista de las fechas faltantes.
df4model = df4model.reindex(fechas_completas)  # Agrega las fechas faltantes con NaN.
df4model['dato'].fillna(method='ffill', inplace=True)  # Rellenar con el último valor válido.

Fechas faltantes: DatetimeIndex(['1991-11-16', '1991-11-17', '1991-11-20', '1991-11-23',
               '1991-11-24', '1991-11-30', '1991-12-01', '1991-12-07',
               '1991-12-08', '1991-12-12',
               ...
               '2025-02-23', '2025-03-01', '2025-03-02', '2025-03-08',
               '2025-03-09', '2025-03-15', '2025-03-16', '2025-03-17',
               '2025-03-22', '2025-03-23'],
              dtype='datetime64[ns]', length=3802, freq=None)


#**Holt-Winter**

In [ ]:
# @title Parámetros del modelo
# @markdown Parámetros generales
Días_a_pronosticar = 90 # @param {"type":"slider","min":0,"max":90,"step":1}
Optimizar = True # @param {"type":"boolean"}
alpha = 0.75 # @param {"type":"slider","min":0,"max":1,"step":0.01}
Frecuencia = "Diaria" # @param ["Diaria","Semanal","Mensual","Trimestral","Anual"]

# @markdown Parámetros de tendencia
Usar_tendencia = True # @param {"type":"boolean"}
if Usar_tendencia :
  Tipo_de_tendencia = "additive" # @param ["additive","multiplicative"]
  beta = 0.01 # @param {"type":"slider","min":0,"max":1,"step":0.01}
  Tendencia_amortiguada = False # @param {"type":"boolean"}
else :
  Tipo_de_tendencia = None
  Tendencia_amortiguada = None
  beta = None

# @markdown Parámetros de estacionalidad
Usar_estacionalidad = True # @param {"type":"boolean"}
if Usar_estacionalidad :
  Tipo_de_estacionalidad = "additive" # @param ["additive","multiplicative"]
  Estaciones = 365 # @param {"type":"integer","placeholder":"2"}
  gamma = 1 # @param {"type":"slider","min":0,"max":1,"step":0.01}
  if Estaciones < 2 :
    Estaciones = None
else :
  Tipo_de_estacionalidad = None
  Estaciones = None
  gamma = None

Frecuencias_dict = {
    "Diaria" : "D",
    "Semanal" : "W",
    "Mensual" : "M",
    "Trimestral" : "Q",
    "Anual" : "A"
} # Diccionario de frecuencias de statsmodels


### Traducciones al inglés de las variables del modelo.
days2forecast = Días_a_pronosticar
trend_type = Tipo_de_tendencia
damped_trend = Tendencia_amortiguada
seasonal_type = Tipo_de_estacionalidad
seasonal_periods = Estaciones
frequency = Frecuencias_dict[Frecuencia]

In [ ]:
# @title Entrenamiento del modelo

if Optimizar :  # Si en los parámetros se seleccionó la opción de optimizar.
  model = ExponentialSmoothing(endog=df4model, trend=trend_type, damped_trend=damped_trend, seasonal=seasonal_type, seasonal_periods=seasonal_periods, freq=frequency).fit(optimized=True)  # Modelo con opción de optimizar.
else :  # Si en los parámetros NO se seleccionó la opción de optimizar.
  model = ExponentialSmoothing(endog=df4model, trend=trend_type, damped_trend=damped_trend, seasonal=seasonal_type, seasonal_periods=seasonal_periods, freq=frequency).fit(smoothing_level=alpha, smoothing_trend=beta, smoothing_seasonal=gamma) # El modelo utilizará los valores asignados para alpha, beta y gamma.

forecast_df = model.predict(start=0, end=len(df4model)+days2forecast-1) # Predice el conjunto de datos original + la cantidad de días asignada en los parámetros.
forecast_df = pd.DataFrame(forecast_df, columns=[0])  # Convierte el forecast en un conjunto de datos, ya que originalmente es una lista.

model_parameters = {
    "alpha" : float(model.params["smoothing_level"]),
    "beta" : float(model.params["smoothing_trend"]),
    "gamma" : float(model.params["smoothing_seasonal"])
} # Diccionario de parámetros, con el fin de mostrar los parámetros utilizados. Útil si se escogió la opción de optimizar.

print(model_parameters) # Muestra el diccionario de parámetros.
forecast_df.head()  # Muestra las primeras filas del conjunto de datos.

{'alpha': 0.9976115158053344, 'beta': 0.0003221179217759897, 'gamma': 0.0001223241494271084}


,0
1991-11-12,3.049858
1991-11-13,3.055374
1991-11-14,3.076792
1991-11-15,3.082597
1991-11-16,3.093467


In [ ]:
# @title Gráfica del conjunto de datos real vs. el pronosticado

real_vs_forecast_plt(real_df = df4model, real_time_column_name = "fecha", real_value_column_name = "dato", forecast_df = forecast_df, title="Tipo de cambio real vs Holt-Winter", model_name_serie="Holt-Winter", yaxis_title = "Tipo de cambio", forecast_value_column_name = 0)  # Crea la gráfica comparativa de las ventas reales vs el pronóstico.

#**MAPE**

In [ ]:
mape_value = MAPE(df4model["dato"], forecast_df.iloc[:-days2forecast][0]) * 100 # Función mean_absolute_percentage_error (MAPE) de sklearn
mape_value  # Muestra el MAPE.

0.38315355086860414

#**Descarga de datos**

In [ ]:
forecast_df.to_csv("forecast_HoltWinter.csv") # Exporta el conjunto de datos de la predicción aun archivo csv.
files.download("forecast_HoltWinter.csv") # Descarga el archivo localmente.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>